# Superyacht HVAC Concept Sizing — Conventional Chilled-Water System

Internal concept sizing calc. Chilled-water plant, fresh-air AHU, local FCUs, N+1 chillers.
All geometry/loads below are synthetic placeholder inputs, not a yacht design.


In [1]:
# %pip install hvacpy==0.4.1

In [2]:
from IPython.display import display, Markdown
from hvacpy import ( 
    Q_, Material, Assembly, AirState, AirProcess, Room, Zone,
    WallComponent, WindowComponent, InternalGain, Orientation,
    CoolingLoad, HeatingLoad, FanCoilUnit, Chiller, DuctSizer,
)

def md_table(headers, rows):
    header = "| " + " | ".join(headers) + " |"
    rule = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = "\n".join("| " + " | ".join(str(v) for v in r) + " |" for r in rows)
    display(Markdown("\n".join([header, rule, body])))

def q(v, unit, nd=2):
    return f"{v.to(unit).magnitude:.{nd}f} {unit}"


In [3]:
# --- Design basis ---
SUMMER_DB, SUMMER_WB = Q_(35, "degC"), Q_(28, "degC") # Summer design
SUMMER_DIURNAL = Q_(8, "delta_degC")  # Diurnal range used by CLTD calculation
INDOOR_T, INDOOR_RH = Q_(22, "degC"), 0.50 # Indoor
WINTER_DB, WINTER_WIND = Q_(0, "degC"), Q_(8, "m/s") # Winter design
CHW_SUPPLY, CHW_RETURN = Q_(6, "degC"), Q_(12, "degC")  # Chilled water: 6/12 °C
FRESH_AIR_LPS_PP = 10.0 # 10 L/s per design occupant of fresh air
CHILLER_COP = 5.5 
N_CHILLERS = 3


In [4]:
# --- Envelope ---
aluminium_skin = Material("Aluminium skin", conductivity=Q_(205, "W/(m*K)"),
                           density=Q_(2700, "kg/m**3"), specific_heat=Q_(900, "J/(kg*K)"),
                           category="metal", source="ASHRAE HOF 2021 Table 4")
marine_insulation = Material("Marine mineral wool", conductivity=Q_(0.035, "W/(m*K)"),
                              density=Q_(45, "kg/m**3"), specific_heat=Q_(1000, "J/(kg*K)"),
                              category="insulation", source="Manufacturer datasheet (assumed)")
interior_panel = Material("Interior composite panel", conductivity=Q_(0.15, "W/(m*K)"),
                           density=Q_(700, "kg/m**3"), specific_heat=Q_(1300, "J/(kg*K)"),
                           category="finish", source="Manufacturer datasheet (assumed)")

hull_assembly = (Assembly("Exterior wall", orientation="wall")
                 .add_layer(aluminium_skin, Q_(6, "mm"))
                 .add_layer(marine_insulation, Q_(100, "mm"))
                 .add_layer(interior_panel, Q_(12, "mm")))

deck_assembly = (Assembly("Exposed deck", orientation="roof")
                 .add_layer(aluminium_skin, Q_(8, "mm"))
                 .add_layer(marine_insulation, Q_(120, "mm"))
                 .add_layer(interior_panel, Q_(12, "mm")))

md_table(["Assembly", "U [W/m2K]", "R [m2K/W]"], [
    ["Exterior wall", q(hull_assembly.u_value, "W/(m**2*K)", 3), q(hull_assembly.r_value, "m**2*K/W", 3)],
    ["Exposed deck", q(deck_assembly.u_value, "W/(m**2*K)", 3), q(deck_assembly.r_value, "m**2*K/W", 3)],
])


| Assembly | U [W/m2K] | R [m2K/W] |
| --- | --- | --- |
| Exterior wall | 0.322 W/(m**2*K) | 3.107 m**2*K/W |
| Exposed deck | 0.274 W/(m**2*K) | 3.649 m**2*K/W |

In [5]:
# --- Spaces (synthetic) ---

room_specs = [
    dict(name="Main Salon", area=95, height=2.7, wall_area=32, window_area=22,
         orientation=Orientation.S, occupants=16, activity="seated_quiet",
         lighting_wm2=10, equipment_wm2=8, exposed_deck=True),
    dict(name="Owner Suite", area=50, height=2.6, wall_area=24, window_area=12,
         orientation=Orientation.E, occupants=2, activity="seated_quiet",
         lighting_wm2=8, equipment_wm2=6, exposed_deck=True),
    dict(name="Guest Cabin 1", area=25, height=2.5, wall_area=14, window_area=5,
         orientation=Orientation.E, occupants=2, activity="seated_quiet",
         lighting_wm2=7, equipment_wm2=4, exposed_deck=False),
    dict(name="Guest Cabin 2", area=25, height=2.5, wall_area=14, window_area=5,
         orientation=Orientation.W, occupants=2, activity="seated_quiet",
         lighting_wm2=7, equipment_wm2=4, exposed_deck=False),
    dict(name="Guest Cabin 3", area=25, height=2.5, wall_area=14, window_area=5,
         orientation=Orientation.E, occupants=2, activity="seated_quiet",
         lighting_wm2=7, equipment_wm2=4, exposed_deck=False),
    dict(name="Guest Cabin 4", area=25, height=2.5, wall_area=14, window_area=5,
         orientation=Orientation.W, occupants=2, activity="seated_quiet",
         lighting_wm2=7, equipment_wm2=4, exposed_deck=False),
    dict(name="Bridge", area=35, height=2.5, wall_area=15, window_area=20,
         orientation=Orientation.S, occupants=4, activity="office_work",
         lighting_wm2=8, equipment_wm2=25, exposed_deck=True),
    dict(name="Crew Mess", area=40, height=2.5, wall_area=15, window_area=4,
         orientation=Orientation.N, occupants=8, activity="restaurant",
         lighting_wm2=9, equipment_wm2=10, exposed_deck=False),
]

# for ships we should use port / starboard / forward / aft / overhead instead of constant orientations...

def make_room(s):
    room = Room(name=s["name"], floor_area=Q_(s["area"], "m**2"),
                ceiling_height=Q_(s["height"], "m"), ach_infiltration=0.20, t_indoor=INDOOR_T)
    room.walls.append(WallComponent("Exterior shell", hull_assembly, Q_(s["wall_area"], "m**2"),
                                     s["orientation"], wall_group="A"))
    if s["exposed_deck"]:
        room.walls.append(WallComponent("Exposed deck", deck_assembly, Q_(s["area"], "m**2"),
                                         Orientation.HORIZONTAL, wall_group="G", is_roof=True))
    room.windows.append(WindowComponent("Exterior glazing", Q_(s["window_area"], "m**2"),
                                         s["orientation"], u_factor=Q_(1.4, "W/(m**2*K)"),
                                         shgc=0.35, has_interior_shading=True, frame_fraction=0.15))
    room.internal_gains.extend([
        InternalGain("people", count=s["occupants"], activity=s["activity"], diversity=1.0, clf=1.0),
        InternalGain("lighting", watts_per_m2=s["lighting_wm2"], diversity=1.0, clf=1.0),
        InternalGain("equipment", watts_per_m2=s["equipment_wm2"], diversity=1.0, clf=1.0),
    ])
    return room


rooms = [make_room(s) for s in room_specs]
accommodation_zone = Zone("Accommodation and public spaces", rooms=rooms)

md_table(["Space", "Area", "Occ.", "Glazing", "Orient.", "Deck"], [
    [s["name"], s["area"], s["occupants"], s["window_area"], s["orientation"].value,
     "Y" if s["exposed_deck"] else "N"]
    for s in room_specs
])


| Space | Area | Occ. | Glazing | Orient. | Deck |
| --- | --- | --- | --- | --- | --- |
| Main Salon | 95 | 16 | 22 | S | Y |
| Owner Suite | 50 | 2 | 12 | E | Y |
| Guest Cabin 1 | 25 | 2 | 5 | E | N |
| Guest Cabin 2 | 25 | 2 | 5 | W | N |
| Guest Cabin 3 | 25 | 2 | 5 | E | N |
| Guest Cabin 4 | 25 | 2 | 5 | W | N |
| Bridge | 35 | 4 | 20 | S | Y |
| Crew Mess | 40 | 8 | 4 | N | N |

In [6]:
# --- Peak zone cooling load (CLTD/CLF) ---
zone_cooling = CoolingLoad(accommodation_zone, t_outdoor_db=SUMMER_DB,
                            t_outdoor_wb=SUMMER_WB, diurnal_range=SUMMER_DIURNAL)

md_table(["Result", "Value"], [
    ["Peak total", q(zone_cooling.peak_total, "kW")],
    ["Peak sensible", q(zone_cooling.peak_sensible, "kW")],
    ["Peak latent", q(zone_cooling.peak_latent, "kW")],
    ["SHR", f"{zone_cooling.sensible_heat_ratio:.3f}"],
    ["Peak hour", f"{zone_cooling.peak_hour}:00"],
])
print(zone_cooling.breakdown())


| Result | Value |
| --- | --- |
| Peak total | 17.77 kW |
| Peak sensible | 14.71 kW |
| Peak latent | 3.06 kW |
| SHR | 0.828 |
| Peak hour | 14:00 |

Cooling Load Breakdown — Peak Hour 14:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Wall conduction              2867.9 W   16.1%
  Window conduction            1419.6 W    8.0%
  Solar gain                   2051.5 W   11.5%
  Infiltration (sensible)       718.4 W    4.0%
  Infiltration (latent)         206.1 W    1.2%
  People (sensible)            2380.0 W   13.4%
  People (latent)              2030.0 W   11.4%
  Lighting                     3362.5 W   18.9%
  Equipment (sensible)         1914.5 W   10.8%
  Equipment (latent)            820.5 W    4.6%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TOTAL SENSIBLE              14714.4 W   82.8%
  TOTAL LATENT                 3056.6 W   17.2%
  PEAK TOTAL                  17771.0 W  100.0%
  SHR = 0.828


In [7]:
# --- Room-by-room loads and FCU selection ---
# Note: zone peak != sum of room peak times (different peak hours per orientation).
room_loads, fcu_schedule = {}, []

for room in rooms:
    load = CoolingLoad(room, t_outdoor_db=SUMMER_DB, t_outdoor_wb=SUMMER_WB,
                        diurnal_range=SUMMER_DIURNAL)
    room_loads[room.name] = load
    fcu = FanCoilUnit(load, chilled_water_supply_t=CHW_SUPPLY,
                       chilled_water_return_t=CHW_RETURN, cop_rated=CHILLER_COP)
    fcu_schedule.append([
        room.name, q(load.peak_total, "kW"), f"{load.peak_hour}:00",
        q(fcu.nominal_capacity, "kW"), q(fcu.supply_airflow, "m**3/s", 3),
        q(fcu.chw_flow_rate, "L/s", 3),
    ])

md_table(["Space", "Peak load", "Peak hr", "FCU cap.", "Supply air", "CHW flow"], fcu_schedule)


| Space | Peak load | Peak hr | FCU cap. | Supply air | CHW flow |
| --- | --- | --- | --- | --- | --- |
| Main Salon | 6.32 kW | 14:00 | 7.00 kW | 0.488 m**3/s | 0.251 L/s |
| Owner Suite | 2.60 kW | 10:00 | 3.50 kW | 0.220 m**3/s | 0.104 L/s |
| Guest Cabin 1 | 1.13 kW | 8:00 | 1.20 kW | 0.092 m**3/s | 0.045 L/s |
| Guest Cabin 2 | 1.17 kW | 16:00 | 1.20 kW | 0.095 m**3/s | 0.047 L/s |
| Guest Cabin 3 | 1.13 kW | 8:00 | 1.20 kW | 0.092 m**3/s | 0.045 L/s |
| Guest Cabin 4 | 1.17 kW | 16:00 | 1.20 kW | 0.095 m**3/s | 0.047 L/s |
| Bridge | 3.33 kW | 14:00 | 3.50 kW | 0.260 m**3/s | 0.133 L/s |
| Crew Mess | 2.36 kW | 17:00 | 2.40 kW | 0.145 m**3/s | 0.094 L/s |

In [8]:
# --- Fresh-air AHU: outdoor air -> 22C/50%RH neutral supply ---
total_occupants = sum(s["occupants"] for s in room_specs)
fresh_air_m3s = total_occupants * FRESH_AIR_LPS_PP / 1000.0

outdoor_air = AirState(dry_bulb=SUMMER_DB, wet_bulb=SUMMER_WB)
neutral_supply_air = AirState(dry_bulb=INDOOR_T, rh=INDOOR_RH)
dry_air_mass_flow = fresh_air_m3s / outdoor_air.specific_volume.to("m**3/kg").magnitude

ahu_process = AirProcess(outdoor_air, neutral_supply_air, Q_(dry_air_mass_flow, "kg/s"))
ahu_sensible_kw = abs(ahu_process.sensible_heat.to("kW").magnitude)
ahu_latent_kw = abs(ahu_process.latent_heat.to("kW").magnitude)
ahu_total_kw = abs(ahu_process.total_heat.to("kW").magnitude)
condensate_kgh = abs(ahu_process.moisture_added.to("kg/h").magnitude)

md_table(["AHU result", "Value"], [
    ["Occupants", total_occupants],
    ["Fresh-air flow", f"{fresh_air_m3s:.3f} m3/s"],
    ["Outdoor RH", f"{outdoor_air.rh * 100:.1f} %"],
    ["Outdoor enthalpy", q(outdoor_air.enthalpy, "kJ/kg")],
    ["Sensible", f"{ahu_sensible_kw:.2f} kW"],
    ["Latent", f"{ahu_latent_kw:.2f} kW"],
    ["Total", f"{ahu_total_kw:.2f} kW"],
    ["Condensate", f"{condensate_kgh:.2f} kg/h"],
])


| AHU result | Value |
| --- | --- |
| Occupants | 38 |
| Fresh-air flow | 0.380 m3/s |
| Outdoor RH | 59.1 % |
| Outdoor enthalpy | 89.39 kJ/kg |
| Sensible | 5.51 kW |
| Latent | 13.57 kW |
| Total | 19.51 kW |
| Condensate | 19.53 kg/h |

In [9]:
# --- Combine zone + AHU into plant load ---
class PlantCoolingLoad:
    """Adapter: combines zone + AHU loads into the interface Chiller expects."""

    def __init__(self, zone_load, ahu_sensible_kw, ahu_latent_kw):
        self._sensible = zone_load.peak_sensible.to("kW").magnitude + ahu_sensible_kw
        self._latent = zone_load.peak_latent.to("kW").magnitude + ahu_latent_kw
        self._total = self._sensible + self._latent

    @property
    def peak_sensible(self):
        return Q_(self._sensible, "kW")

    @property
    def peak_latent(self):
        return Q_(self._latent, "kW")

    @property
    def peak_total(self):
        return Q_(self._total, "kW")

    @property
    def sensible_heat_ratio(self):
        return self._sensible / self._total


plant_load = PlantCoolingLoad(zone_cooling, ahu_sensible_kw, ahu_latent_kw)

md_table(["Plant load", "Value"], [
    ["Zone", q(zone_cooling.peak_total, "kW")], # the simultaneous zone cooling load
    ["AHU", f"{ahu_total_kw:.2f} kW"], # the fresh-air AHU duty.
    ["Total", q(plant_load.peak_total, "kW")],
    ["SHR", f"{plant_load.sensible_heat_ratio:.3f}"],
])


| Plant load | Value |
| --- | --- |
| Zone | 17.77 kW |
| AHU | 19.51 kW |
| Total | 36.85 kW |
| SHR | 0.549 |

In [10]:
# --- Chiller sizes each machine so the design load can still be met with one chiller unavailable.
# chiller_plant = Chiller(plant_load, chiller_type="water_cooled", cop_rated=CHILLER_COP,
#                          n_units=N_CHILLERS, redundancy="n+1")


Q_design = plant_load.peak_total.to("kW").magnitude
n_operating = N_CHILLERS - 1    # N+1
required_per_chiller = Q_design / n_operating
design_margin = 1.10
per_unit_kw = required_per_chiller * design_margin


# per_unit_kw = chiller_plant.capacity_per_unit.to("kW").magnitude
installed_kw = N_CHILLERS * per_unit_kw
n_plus_1_kw = (N_CHILLERS - 1) * per_unit_kw
condenser_heat_rejection_kw = Q_design * (1 + 1/CHILLER_COP)

md_table(["Chiller plant", "Value"], [
    ["Required", q(plant_load.peak_total, "kW")],
    ["Units / redundancy", f"{N_CHILLERS}, N+1"],
    ["Capacity / unit", f"{per_unit_kw:.2f} kW"],
    ["Installed", f"{installed_kw:.2f} kW"],
    ["Available, one unit down", f"{n_plus_1_kw:.2f} kW"],
    ["COP", f"{CHILLER_COP:.2f}"],
    ["Condenser heat rejection", f"{condenser_heat_rejection_kw:.2f} kW"],
])


| Chiller plant | Value |
| --- | --- |
| Required | 36.85 kW |
| Units / redundancy | 3, N+1 |
| Capacity / unit | 20.27 kW |
| Installed | 60.80 kW |
| Available, one unit down | 40.53 kW |
| COP | 5.50 |
| Condenser heat rejection | 43.55 kW |

In [11]:
# --- Main fresh-air duct, equal-friction ---
main_duct = DuctSizer(Q_(fresh_air_m3s, "m**3/s"), method="equal_friction")
print(main_duct.summary())


Dia350mm - 3.9m/s - 0.52Pa/m - or 400x300mm rect


In [12]:
# --- Winter heating check, same geometry, no solar/internal credit ---
zone_heating = HeatingLoad(accommodation_zone, t_winter_db=WINTER_DB, wind_speed=WINTER_WIND)

md_table(["Winter result", "Value"], [
    ["Envelope loss", q(zone_heating.envelope_loss, "kW")],
    ["Infiltration loss", q(zone_heating.infiltration_loss, "kW")],
    ["Total", q(zone_heating.total, "kW")],
])


| Winter result | Value |
| --- | --- |
| Envelope loss | 4.49 kW |
| Infiltration loss | 1.33 kW |
| Total | 5.82 kW |

In [13]:
# --- Supplier-proposal check (fictional numbers, demonstrates the workflow) ---
supplier = dict(fresh_air_m3s=0.40, ahu_cooling_kw=42.0, chiller_units=3, chiller_capacity_each_kw=80.0)

checks = [
    ("Fresh-air flow", fresh_air_m3s, supplier["fresh_air_m3s"],
     supplier["fresh_air_m3s"] >= fresh_air_m3s, "m3/s"),
    ("AHU cooling duty", ahu_total_kw, supplier["ahu_cooling_kw"],
     supplier["ahu_cooling_kw"] >= ahu_total_kw, "kW"),
    ("N+1 chiller capacity", plant_load.peak_total.to("kW").magnitude,
     (supplier["chiller_units"] - 1) * supplier["chiller_capacity_each_kw"],
     (supplier["chiller_units"] - 1) * supplier["chiller_capacity_each_kw"] >= plant_load.peak_total.to("kW").magnitude,
     "kW"),
]

md_table(["Check", "Required", "Supplier", "Status"], [
    [name, f"{req:.2f} {unit}", f"{prop:.2f} {unit}", "PASS" if ok else "FAIL"]
    for name, req, prop, ok, unit in checks
])


| Check | Required | Supplier | Status |
| --- | --- | --- | --- |
| Fresh-air flow | 0.38 m3/s | 0.40 m3/s | PASS |
| AHU cooling duty | 19.51 kW | 42.00 kW | PASS |
| N+1 chiller capacity | 36.85 kW | 160.00 kW | PASS |

In [14]:
# --- Summary ---
md_table(["Design output", "Result"], [
    ["Zone peak cooling", q(zone_cooling.peak_total, "kW")],
    ["AHU cooling", f"{ahu_total_kw:.2f} kW"],
    ["Plant load", q(plant_load.peak_total, "kW")],
    ["Peak hour", f"{zone_cooling.peak_hour}:00"],
    ["Fresh-air flow", f"{fresh_air_m3s:.3f} m3/s"],
    ["Condensate", f"{condensate_kgh:.2f} kg/h"],
    ["Chillers", f"{N_CHILLERS}, N+1, {per_unit_kw:.2f} kW/unit"],
    ["N+1 available", f"{n_plus_1_kw:.2f} kW"],
    ["Winter heating load", q(zone_heating.total, "kW")],
])


| Design output | Result |
| --- | --- |
| Zone peak cooling | 17.77 kW |
| AHU cooling | 19.51 kW |
| Plant load | 36.85 kW |
| Peak hour | 14:00 |
| Fresh-air flow | 0.380 m3/s |
| Condensate | 19.53 kg/h |
| Chillers | 3, N+1, 20.27 kW/unit |
| N+1 available | 40.53 kW |
| Winter heating load | 5.82 kW |